# SchoolBridge — LiLT Sentence Grouping PoC (B안)

**목적**: 구강검진 PDF 1장 라벨링 → Mini fine-tune → 같은 페이지 추론. LiLT가 한국어 통신문에서 의미 묶음을 학습할 능력이 있는지 빠르게 검증.

**핵심 검증 포인트** — 진심담은치과 vs 청담i치과 정보가 페이지에서 행 단위로 인터리브되어 있는데, 모델이 **같은 열끼리 묶어서** 별도 sentence로 잡아내는지.

**소요**: ~30분 (라벨링 15분 + fine-tune 5분 + 추론 1분)

**Colab 세팅**: 런타임 → T4 GPU

## 1. 환경 + 설치

In [ ]:
!pip install -q transformers sentencepiece pdfplumber pymupdf pillow
print("설치 완료")

In [ ]:
import torch
from transformers import AutoTokenizer, LiltForTokenClassification
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("LiLT import OK")

## 2. PDF 업로드

구강검진 PDF 업로드 — `2026 2,3,5,6학년 구강검진 실시안내.pdf`

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
pdf_paths = [Path(name) for name in uploaded.keys() if name.lower().endswith(".pdf")]
for p in pdf_paths:
    print(f"  {p.name}: {p.stat().st_size // 1024} KB")

## 3. PDF에서 토큰 + bbox 추출

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import pdfplumber

@dataclass
class TextSpan:
    text: str
    bbox: Tuple[float, float, float, float]
    page: int = 0

def extract_pdf(path: Path) -> List[TextSpan]:
    spans = []
    with pdfplumber.open(path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            W, H = page.width, page.height
            words = page.extract_words(
                use_text_flow=True,
                keep_blank_chars=False,
                x_tolerance=3,
                y_tolerance=3,
            )
            for w in words:
                bbox = (w["x0"]/W, w["top"]/H, w["x1"]/W, w["bottom"]/H)
                spans.append(TextSpan(text=w["text"], bbox=bbox, page=page_idx))
    return spans

sample_pdf = pdf_paths[0]
all_spans = extract_pdf(sample_pdf)
page_spans = [s for s in all_spans if s.page == 0]
print(f"📄 {sample_pdf.name}: {len(all_spans)} tokens (page 0: {len(page_spans)})")

## 4. LiLT 모델 로드

`SCUT-DLVCLab/lilt-infoxlm-base` — InfoXLM 토크나이저 (한국어 강함). `MAX_SENT_ID = 20` — 페이지당 최대 20개 의미 묶음.

In [ ]:
MODEL_ID = "SCUT-DLVCLab/lilt-infoxlm-base"
MAX_SENT_ID = 20

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = LiltForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=MAX_SENT_ID,
    id2label={i: f"SENT_{i}" for i in range(MAX_SENT_ID)},
    label2id={f"SENT_{i}": i for i in range(MAX_SENT_ID)},
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params | Device: {device}")

## 5. 입력 변환 함수

In [ ]:
def to_lilt_inputs(spans, word_labels=None):
    words = [s.text for s in spans]
    boxes = []
    for s in spans:
        x0, y0, x1, y1 = s.bbox
        boxes.append([
            max(0, min(1000, int(x0 * 1000))),
            max(0, min(1000, int(y0 * 1000))),
            max(0, min(1000, int(x1 * 1000))),
            max(0, min(1000, int(y1 * 1000))),
        ])
    kwargs = dict(
        boxes=boxes,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
        is_split_into_words=True,
    )
    if word_labels is not None:
        kwargs["word_labels"] = word_labels
    return tokenizer(words, **kwargs)

print("to_lilt_inputs 정의 완료")

## 6. 라벨링용 spans 확인 — 인덱스 + 텍스트 출력

이 출력 보면서 다음 셀에서 sentence_ranges 작성.

In [ ]:
print(f"=== {sample_pdf.name} — page_spans ({len(page_spans)}개) ===\n")
for i, s in enumerate(page_spans):
    y = s.bbox[1]
    print(f"  [{i:3d}] (y={y:.3f}) {s.text!r}")

## 7. 라벨링 — sentence_ranges 작성

**위 셀 6 출력을 보고 같은 의미 묶음 토큰들의 index 범위를 부여**.

**핵심**: 진심담은치과 정보 묶음과 청담i치과 정보 묶음을 **별도 sentence_id로 분리** — 페이지 시각적으로는 행 단위 인터리브되어 있어도.

아래는 예시 — 실제 index는 셀 6 출력 보고 수정해주세요.

In [ ]:
from collections import Counter

# (start_idx, end_idx_inclusive, sentence_id, "주석")
# 위 셀 6 출력 보고 정확한 index로 수정
sentence_ranges = [
    (0,   9,   0, "발송 정보 — 가능교육통신·발송처·교무실·전화"),
    (10,  15,  1, "통신문 제목 + 인사"),
    (16,  60,  2, "본문 안내문"),
    (61,  75,  3, "1. 검진 대상"),
    (76,  95,  4, "2. 검진 기간"),
    (96,  110, 5, "3. 검진 항목"),
    (111, 125, 6, "4. 검사 비용"),
    (126, 145, 7, "5. 검진 기관"),
    
    # ===== 표 영역 — 핵심 검증 =====
    # 진심담은치과 묶음 (시각적으로 청담i치과와 인터리브)
    (146, 175, 10, "진심담은치과의원 — 이름·전화·주소·시간 묶음"),
    # 청담i치과 묶음
    (176, 205, 11, "청담i치과의원 — 이름·전화·주소·시간 묶음"),
    
    (206, 225, 12, "발급일자 / 학교장 / 절취선"),
    (226, 230, 13, "구강검진 확인서 양식"),
]

# word_labels 생성
word_labels = [-1] * len(page_spans)
for start, end, sid, note in sentence_ranges:
    for i in range(start, min(end + 1, len(word_labels))):
        word_labels[i] = sid

# 미라벨링 → 별도 id (마지막 라벨 + 1)
max_sid = max(sid for _, _, sid, _ in sentence_ranges)
UNLABELED_ID = max_sid + 1
word_labels = [w if w >= 0 else UNLABELED_ID for w in word_labels]

print(f"라벨링 완료")
print(f"  총 토큰: {len(word_labels)}")
print(f"  의미 묶음: {len(set(word_labels))}개\n")
print("라벨 분포:")
for sid, cnt in sorted(Counter(word_labels).items()):
    note = next((n for _s, _e, _sid, n in sentence_ranges if _sid == sid), "미라벨")
    print(f"  id={sid:2d}: {cnt:3d}개  — {note}")

## 8. Mini Fine-tune (1장 overfit, 30 epoch)

In [ ]:
encoded = to_lilt_inputs(page_spans, word_labels=word_labels)
inputs = {k: v.to(device) for k, v in encoded.items()}

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

print("=== Mini fine-tune (1 page overfit, 30 epoch) ===")
for epoch in range(30):
    optimizer.zero_grad()
    outputs = model(**inputs)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    if epoch % 3 == 0 or epoch == 29:
        print(f"  Epoch {epoch+1:2d}: loss = {loss.item():.4f}")

print("\n학습 완료. Loss가 1.0 → 0.1 이하로 떨어지면 학습 정상.")

## 9. 추론 + sentence_list 생성

같은 페이지에 추론 (overfit이라 90%+ 정확도 기대). 핵심 — 진심담은치과 vs 청담i치과가 별도 묶음으로 분리되는지.

In [ ]:
from collections import defaultdict

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
predictions = outputs.logits.argmax(-1)[0].tolist()

# 토큰 → 단어 voting
word_ids = encoded.word_ids() if hasattr(encoded, "word_ids") else None
word_preds = []
if word_ids:
    for w_idx in range(len(page_spans)):
        tok_preds = [predictions[t] for t, w in enumerate(word_ids) if w == w_idx]
        if tok_preds:
            word_preds.append(Counter(tok_preds).most_common(1)[0][0])
        else:
            word_preds.append(-1)

# 정확도
correct = sum(1 for p, g in zip(word_preds, word_labels) if p == g)
acc = correct / len(word_preds) * 100
print(f"=== 정확도 (overfit) ===")
print(f"  {correct} / {len(word_preds)} = {acc:.1f}%")
print(f"  → 90%+ 기대 (학습 가능성 검증)")

# 틀린 예측
errors = [(i, span, p, g) for i, (span, p, g) in enumerate(zip(page_spans, word_preds, word_labels)) if p != g]
if errors:
    print(f"\n=== 틀린 예측 (처음 20개 / 총 {len(errors)}개) ===")
    for i, span, pred, gt in errors[:20]:
        print(f"  ❌ [{i:3d}] pred={pred:3d} gt={gt:3d}  {span.text!r}")
else:
    print("\n✅ 전부 정확!")

# sentence_list 생성
groups = defaultdict(list)
first_app = {}
for idx, (span, sid) in enumerate(zip(page_spans, word_preds)):
    groups[sid].append(span)
    first_app.setdefault(sid, idx)
sorted_sids = sorted(groups.keys(), key=lambda s: first_app[s])

print(f"\n=== 추론된 sentence_list ({len(sorted_sids)}개 묶음) ===")
for sid in sorted_sids:
    text = " ".join(s.text for s in groups[sid])
    note = next((n for _s, _e, _sid, n in sentence_ranges if _sid == sid), "")
    print(f"\n[묶음 id={sid}]  ({note})")
    print(f"  {text[:200]}")

## 10. 결과 해석

### 검증 포인트

1. **정확도 90%+** ← Mini fine-tune 학습 능력 검증
2. **진심담은치과 묶음 (id=10) ≠ 청담i치과 묶음 (id=11)** ← 시각 인터리브 ↔ 의미 묶음 분리 검증
3. **본문 항목 (1. 검진 대상, 2. 검진 기간 등) 각각 별도 묶음** ← 순차 의미 분리 검증

### 결과 해석 가이드

| 결과 | 의미 |
|---|---|
| 정확도 95%+ | 모델 학습 능력 OK — 라벨링 데이터만 늘리면 일반화 가능 |
| 두 치과 묶음 정상 분리 | **bbox 좌표만으로 표 의미 결합 가능** — 라벨링 300~500장 투자 가치 |
| 두 치과 묶음 혼합 (id 같음) | LiLT가 시각 신호 없이 부족 — LayoutXLM (image 포함) 고려 |
| 정확도 70% 이하 | 라벨링 정확도 의심 또는 모델 부적합 |

### 다음 단계

- 결과 좋으면: Label Studio 셋업 → 다른 통신문 100~200장 라벨링 → 본격 fine-tune
- 결과 나쁘면: LayoutXLM (image 포함) 또는 다단계 학습 검토